# PathFinder LK — Architecture Design & Agent Testing
**Phase 2 milestone:** architecture + at least one working agent, tested in Colab.

## Problem
Independent travellers in Sri Lanka lack a single grounded planning tool that combines local seasonal knowledge, transport quirks and attraction logistics. PathFinder LK is a multi-agent, RAG-grounded travel assistant.

## Architecture
```
User → Orchestrator → Router Agent (Llama 3.1 8B)
                        │ intent
                        ├─ itinerary_planning → Planner Agent (8B) → sub-queries
                        └─ destination_info  ─────────────────────────┐
                                                                      ▼
                     Retrieval Agent (Chroma + 8B re-rank) → Synthesis Agent (70B)
                                                              draft → critique → revise
```

## Agentic patterns
1. **Router** — intent classification gate (tested below)
2. **Planning / task decomposition** — itinerary → sub-queries
3. **Tool use** — vector store retrieval + LLM re-ranking
4. **Reflection** — synthesis self-critique loop

## Agent-to-agent communication
All hops exchange structured `AgentMessage` objects (custom A2A-inspired protocol, defined below).

In [1]:
# Install dependencies
%pip -q install groq pydantic

In [2]:
# Secure key handling — never hard-code the key in the notebook.
# In Colab: click the key icon (left sidebar) -> add secret GROQ_API_KEY, or run this cell to paste it hidden.
import os
from getpass import getpass

try:
    from google.colab import userdata  # Colab secrets, if set
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    if not os.environ.get("GROQ_API_KEY"):
        os.environ["GROQ_API_KEY"] = getpass("Paste GROQ_API_KEY: ")

print("Key loaded:", bool(os.environ.get("GROQ_API_KEY")))

Paste GROQ_API_KEY: ··········
Key loaded: True


## 1. The message protocol
Every agent communicates via `AgentMessage` — carrying a `trace_id` (follow one query across agents), a `performative` (speech-act type: request / inform / propose / critique / final / reject) and a typed `content` payload.

In [3]:
import uuid
from datetime import datetime, timezone
from typing import Any, Literal, Optional
from pydantic import BaseModel, Field

Performative = Literal["request", "inform", "propose", "critique", "final", "reject"]

class AgentMessage(BaseModel):
    trace_id: str
    sender: str
    receiver: str
    performative: Performative
    intent: Optional[str] = None
    content: dict[str, Any] = Field(default_factory=dict)
    timestamp: str = Field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

    @classmethod
    def new_trace(cls, **kw):
        return cls(trace_id=str(uuid.uuid4()), **kw)

    def reply(self, sender, performative, content, intent=None):
        return AgentMessage(trace_id=self.trace_id, sender=sender, receiver=self.sender,
                            performative=performative, intent=intent or self.intent, content=content)

print("Protocol defined ✅")

Protocol defined ✅


## 2. Working agent: the Router
Model choice: **Llama 3.1 8B Instant (Groq)** — routing is a cheap 3-way classification on every request's critical path, so we want minimum latency and cost, not deep reasoning.

In [4]:
import json, re
from groq import Groq

client = Groq()
FAST_MODEL = "llama-3.1-8b-instant"

ROUTER_SYSTEM = '''You are the routing component of PathFinder LK, a Sri Lanka travel
assistant. Classify the user's query into exactly one intent:
- "destination_info": factual questions about a place, ticket, transport, season or activity in Sri Lanka.
- "itinerary_planning": the user wants a multi-stop or multi-day trip plan.
- "out_of_scope": anything not related to travel in Sri Lanka.
Respond ONLY with JSON: {"intent": "...", "reason": "one short sentence"}'''

def route(msg: AgentMessage) -> AgentMessage:
    raw = client.chat.completions.create(
        model=FAST_MODEL, temperature=0.0,
        messages=[{"role": "system", "content": ROUTER_SYSTEM},
                  {"role": "user", "content": msg.content["query"]}],
    ).choices[0].message.content
    m = re.search(r"\{.*\}", raw, re.DOTALL)
    result = json.loads(m.group(0)) if m else {"intent": "out_of_scope", "reason": "parse fail"}
    return msg.reply(sender="router", performative="inform",
                     intent=result.get("intent", "out_of_scope"),
                     content={"query": msg.content["query"], "reason": result.get("reason", "")})

print("Router agent ready ✅")

Router agent ready ✅


In [5]:
# Test the router with structured agent-to-agent messages (+ latency)
import time

test_queries = [
    "What time does Sigiriya open and how much is the ticket?",
    "Plan me 4 days covering the hill country and the south coast",
    "Write me a Python bubble sort",
    "Best month for whale watching in Mirissa?",
    "mama yanna one ella walata, plan ekak hadala denna",   # Singlish itinerary request
]

latencies = []
for q in test_queries:
    user_msg = AgentMessage.new_trace(sender="user", receiver="router",
                                      performative="request", content={"query": q})
    t0 = time.time()
    routed = route(user_msg)
    ms = (time.time() - t0) * 1000
    latencies.append(ms)
    print(f"{routed.intent:20s} | {ms:6.0f} ms | {q}")
    print(f"{'':20s} |           | reason: {routed.content['reason']}\n")

print(f"Average latency: {sum(latencies)/len(latencies):.0f} ms")

destination_info     |    273 ms | What time does Sigiriya open and how much is the ticket?
                     |           | reason: query about opening time and ticket price of a Sri Lankan destination

itinerary_planning   |    160 ms | Plan me 4 days covering the hill country and the south coast
                     |           | reason: user wants a multi-stop or multi-day trip plan covering specific regions in Sri Lanka.

out_of_scope         |    514 ms | Write me a Python bubble sort
                     |           | reason: parse fail

destination_info     |    267 ms | Best month for whale watching in Mirissa?
                     |           | reason: query about a specific activity in a Sri Lankan location

itinerary_planning   |    172 ms | mama yanna one ella walata, plan ekak hadala denna
                     |           | reason: user is asking for a trip plan

Average latency: 277 ms
